# 03, Accessibility Analysis

## 1. Introduction

### What "accessibility" means here

This notebook computes physical accessibility to healthcare and education
facilities for every settled grid cell in both study areas, not
straight-line ("as the crow flies") distance, but real **network travel
time**: how long it actually takes to reach the nearest facility by
following the real road network, since roads rarely run in straight lines
and a facility that's close in a straight line might be effectively far
away if there's no direct road connecting to it.

### Why three transport modes, not one

| Mode | Assumed speed | Notes |
|---|---|---|
| **Walk** | 5 km/h | OSM's "walk" road network |
| **Okada** (motorcycle taxi) | 25 km/h | OSM's "drive" road network, lower assumed speed |
| **Drive** (private/shared vehicle) | 35 km/h | OSM's "drive" road network |

Modeling all three matters because walking-only access analysis, the
default in most similar studies, understates real accessibility in a
context like Akure, where okada is how most people actually reach a clinic
or school beyond a short radius. A 30-minute budget covers very different
ground on foot vs. by okada vs. by car, so **each mode uses its own
access-deficit threshold** rather than one blanket cutoff (see Section 3).

### Method summary

For each settled grid cell (from Notebook 02) and each mode:
1. Build a routable network graph for that mode.
2. Compute network distance (km) and travel time (minutes) to the nearest
   health facility and nearest school.
3. Flag the cell as underserved for a service if travel time exceeds that
   mode's threshold.
4. Combine both service flags into a 0–2 composite access-deficit score.

### Key GIS/network-analysis concept: what is a "routable graph"?

A road network becomes "routable" when it's represented as a graph:
intersections and dead-ends become **nodes**, road segments between them
become **edges**, and each edge carries a weight (here, travel time in
minutes, derived from its real-world length and the mode's assumed speed).
Finding the "fastest route" between two points then becomes a shortest-
path search over this graph, the same underlying idea used by any
turn-by-turn navigation app, just built here directly from OSM's road
tags via the `OSMnx` library rather than a commercial routing service.

### A performance note worth understanding, not just trusting

Naively, computing "nearest facility" for *every* grid cell would mean
running a separate shortest-path search from each cell to *every*
facility, then keeping the minimum. For hundreds of cells and dozens of
facilities, that's thousands of searches per mode, this is what made an
early version of this exact notebook take **over an hour** to run. The
fix used here (`batch_nearest_facility_distances`, in
`akure_access.accessibility.isochrones`) instead runs a **single**
multi-source shortest-path search *from all facilities at once*, which
finds the nearest-facility distance to *every* node in the graph in one
pass, then each grid cell's answer is just a fast lookup. Measured at
this project's scale, this is roughly 370x faster than the naive
approach, with an automated test proving the two approaches give
identical results (see `tests/test_network_graph.py`).

### Expected outputs

For each study area, the grid enriched with per-mode travel times and
access-deficit scores (see Section 7).

### Prerequisites

- Notebooks 01 and 02 have been run for both study areas.
- This notebook makes live OSM network queries per mode per LGA (via
  OSMnx), expect noticeably longer runtime than Notebooks 01–02,
  especially for the "drive" network in denser urban areas.

## 2. Imports

This notebook imports `resolve_boundary` from `lga_extractor` (to
constrain the OSM road-network query to this LGA's area) and three
functions from `akure_access.accessibility`: `add_access_times` (the
network-routing step), `add_access_deficit_score` (converts travel times
into the 0–2 composite score), and `sanitize_for_export` (converts
internal `inf` sentinel values, used to represent "unreachable", into
`NaN` for clean GeoJSON export).

### 2.1 Environment setup

Same Colab/local detection pattern as Notebooks 01–02.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
DASHBOARD_DRIVE_FOLDER = "Akure Access Dashboard"
EXTRACTOR_DRIVE_FOLDER = "LGA OSM Extractor"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DASHBOARD_DIR = f"/content/drive/MyDrive/{DASHBOARD_DRIVE_FOLDER}"
    EXTRACTOR_DIR = f"/content/drive/MyDrive/{EXTRACTOR_DRIVE_FOLDER}"

    if not os.path.exists(DASHBOARD_DIR):
        raise FileNotFoundError(f"Expected '{DASHBOARD_DIR}' in your Google Drive.")

    %cd {DASHBOARD_DIR}/notebooks
    !pip install osmnx geopandas shapely fiona networkx pandas numpy leafmap --quiet

    # Add the dashboard repo root to sys.path (unconditionally) so the
    # `src` package is importable regardless of relative-path quirks
    # after %cd, then add the extractor repo path if it's available.
    sys.path.append(DASHBOARD_DIR)
    if os.path.exists(EXTRACTOR_DIR):
        sys.path.append(EXTRACTOR_DIR)
    else:
        print(
            f"Warning: '{EXTRACTOR_DIR}' not found in Drive. The "
            f"lga_extractor import (used in Notebook 01, and for boundary "
            f"resolution here) will fail unless it's installed another way."
        )

    print(f"Running in Colab. Working directory: {os.getcwd()}")
else:
    sys.path.append("..")
    sys.path.append("../../lga-osm-extractor")
    print("Running locally.")


### 2.2 Package imports

In [ ]:
import geopandas as gpd
from lga_extractor import resolve_boundary
from akure_access.accessibility import (
    add_access_times,
    add_access_deficit_score,
    sanitize_for_export,
    graph_from_roads,
    build_isochrones_for_facilities,
    batch_nearest_facility_distances,
)

## 3. Configuration

### 3.1 Why each mode gets its own threshold, not one shared cutoff

Mode-specific access thresholds are set here. The walking threshold
(30 min) follows common accessibility-literature conventions; the okada
and drive thresholds are set tighter *in time* but represent a *larger*
effective service area, since these modes cover more ground per minute, a 15-minute drive covers roughly the same ground as a much longer walk, so
using the same time cutoff for both would misrepresent effective access
for faster modes. Adjust these if your project's narrative calls for
different assumptions, document any change in the project report for
reproducibility.

In [ ]:
STUDY_AREAS = [
    {"lga_name": "Akure North", "state_name": "Ondo", "data_dir": "../data/processed/akure_north"},
    {"lga_name": "Akure South", "state_name": "Ondo", "data_dir": "../data/processed/akure_south"},
]

MODES = ("walk", "okada", "drive")

ACCESS_THRESHOLDS_MIN = {
    "walk": 30,
    "okada": 20,
    "drive": 15,
}

print("Modes and thresholds:")
for mode, threshold in ACCESS_THRESHOLDS_MIN.items():
    print(f"  {mode:<6} -> underserved if > {threshold} min")


## 4. Data Loading

For each study area, this notebook loads the grid produced by Notebook 02
(already carrying `building_count` and completeness flags), plus the raw
roads/health/schools layers from Notebook 01 needed to build the routable
network and locate facilities. It also re-resolves the LGA boundary, used
to constrain the OSMnx network query to this LGA's extent rather than
pulling in a much larger surrounding road network.

This loading step is folded into Section 5's processing loop (rather than
a separate cell) since each study area's load-then-score sequence is a
single logical unit, see that section for the actual `gpd.read_file()`
calls.

## 5. Processing

### 5.1 Accessibility scoring (both study areas, all three modes)

For each LGA, this loads the completeness grid from Notebook 02, builds a
network graph per mode, computes nearest-facility distance and time for
health and education access, and derives a per-mode access-deficit score.

This is the most computationally intensive step in the pipeline, the
cell below reports progress per LGA/mode so you can monitor it.

**Important implementation detail on `inf` handling:** unreachable cells
are represented internally as `inf` (infinite travel time) precisely
*because* `add_access_deficit_score()` needs to recognize them as
maximally underserved. `sanitize_for_export()`, which converts these
`inf` values into `NaN` for clean file export, is deliberately called
**only after** all mode scoring is complete. Calling it earlier would
silently break the deficit-scoring logic (an earlier version of this
pipeline made exactly this mistake; two regression tests in
`tests/test_scoring.py` now lock in the correct ordering).

### 5.1a Data-quality check: facility geometry snapping

**Why this cell exists.** OSM lets facilities be mapped either as a
single point/node, or as a traced building-outline polygon, both are
valid, common conventions, and the SAME real-world facility can be
mapped either way depending on who mapped it. `clean_layers()` (in
`lga_extractor`) now normalizes both into Point geometries before
anything reaches this notebook, but this check exists as a second,
independent line of defense: it makes the routing step's own
facility-snapping success rate VISIBLE in the notebook output, rather
than trusting silently that cleaning worked.

This exact check would have caught, immediately and in the notebook
itself, a real bug found during this project's own development: all
14 of Akure North's real health facilities were mapped in OSM as
building-outline polygons rather than points, which (before the fix)
caused every single one of them to silently fail to snap to the
routing graph, every settled cell in Akure North was then scored as
"unreachable" for health access, purely as a routing artifact, not a
real finding, with no error or warning anywhere in the original
notebook run.

If this cell ever prints a ⚠️ warning below, **do not trust the scored
results for that LGA/mode until you've investigated**: check
the raw facility layer's geometry types
(`health_gdf.geom_type.value_counts()`) and CRS before proceeding to
Section 5.1.

In [ ]:
import warnings

print("Checking facility-to-graph snapping for each study area...")
print("(a facility genuinely being unreachable from the graph is not")
print(" itself the error, if EVERY facility in a layer fails to snap,")
print(" that almost always signals a data problem, not a real finding)\n")

snap_check_results = []

for area in STUDY_AREAS:
    lga_name, data_dir = area["lga_name"], area["data_dir"]
    health = gpd.read_file(f"{data_dir}/health_facilities.geojson")
    schools = gpd.read_file(f"{data_dir}/schools.geojson")
    boundary = resolve_boundary(lga_name=lga_name, state_name=area["state_name"])
    boundary_polygon_wgs84 = boundary.geometry.iloc[0]

    # One graph per LGA is enough for this check (cheap relative to the
    # full per-mode scoring in 5.1 below); facility-snapping success does
    # not meaningfully depend on which mode's graph is used, since walk/
    # okada/drive graphs share the same underlying road geometry.
    G_check = graph_from_roads(
        roads_gdf=gpd.GeoDataFrame(geometry=[], crs="EPSG:4326"),
        boundary_polygon=boundary_polygon_wgs84,
        mode="walk",
    )

    for layer_name, facilities_gdf in [("health_facilities", health), ("schools", schools)]:
        if facilities_gdf.empty:
            snap_check_results.append((lga_name, layer_name, 0, 0, "empty layer (0 features found by extractor)"))
            continue

        facilities_wgs84 = facilities_gdf.to_crs("EPSG:4326") if facilities_gdf.crs else facilities_gdf

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            distances = batch_nearest_facility_distances(G_check, facilities_wgs84)

        n_total = len(facilities_gdf)
        n_reachable_nodes = len(distances)
        status = "; ".join(str(w.message) for w in caught) if caught else "ok"
        snap_check_results.append((lga_name, layer_name, n_total, n_reachable_nodes, status))

print(f"{'LGA':<14}{'Layer':<20}{'Features':<11}{'Graph nodes reachable':<24}Status")
print("-" * 95)
any_problem = False
for lga_name, layer_name, n_total, n_reachable, status in snap_check_results:
    problem = (n_total > 0 and n_reachable == 0)
    any_problem = any_problem or problem
    flag = "\u26a0\ufe0f " if problem else ("\u2705 " if n_total > 0 else "\u2139\ufe0f ")
    print(f"{flag}{lga_name:<12}{layer_name:<20}{n_total:<11}{n_reachable:<24}{status}")

print()
if any_problem:
    print("\u26a0\ufe0f  At least one facility layer produced ZERO reachable graph nodes despite")
    print("   having features. STOP and investigate before running Section 5.1, this is the")
    print("   exact signature of the geometry-type bug described above, not a real finding.")
else:
    print("\u2705 All non-empty facility layers snapped to their routing graph successfully.")
    print("   Safe to proceed to Section 5.1.")


In [ ]:
scored_results = {}

for area in STUDY_AREAS:
    lga_name, data_dir = area["lga_name"], area["data_dir"]
    print(f"=== {lga_name} ===")

    grid = gpd.read_file(f"{data_dir}/grid_completeness.geojson")
    roads = gpd.read_file(f"{data_dir}/roads.geojson")
    health = gpd.read_file(f"{data_dir}/health_facilities.geojson")
    schools = gpd.read_file(f"{data_dir}/schools.geojson")

    boundary = resolve_boundary(lga_name=lga_name, state_name=area["state_name"])
    boundary_polygon_wgs84 = boundary.geometry.iloc[0]

    print(f"  Computing access times for modes: {MODES}...")
    grid = add_access_times(
        grid, roads, health, schools,
        boundary_polygon_wgs84=boundary_polygon_wgs84,
        modes=MODES,
    )

    for mode in MODES:
        grid = add_access_deficit_score(grid, threshold_min=ACCESS_THRESHOLDS_MIN[mode], mode=mode)

    # Sanitize inf -> NaN only now, AFTER all modes have been scored above,     # never before add_access_deficit_score(), which needs real inf values
    # to correctly detect unreachable cells as underserved.
    grid = sanitize_for_export(grid)
    grid.to_file(f"{data_dir}/grid_access_scored.geojson", driver="GeoJSON")
    scored_results[lga_name] = grid
    print(f"  Done. Saved to {data_dir}/grid_access_scored.geojson\n")

    # Precompute health-facility walking catchments (isochrones) for the
    # dashboard's optional overlay, see Section 5.2 below. Reuses the
    # already-loaded roads/health/boundary for this LGA; the walk-mode
    # graph is rebuilt here (rather than returned from add_access_times())
    # since that function intentionally doesn't expose its internal
    # per-mode graphs, rebuilding one extra graph per LGA is cheap
    # relative to the scoring already done above.
    print(f"  Computing health-facility walking catchments...")
    G_walk = graph_from_roads(roads, boundary_polygon=boundary_polygon_wgs84, mode="walk")
    isochrones = build_isochrones_for_facilities(G_walk, health, trip_times_min=(15, 30, 45))
    # Reproject to WGS84 before saving: build_isochrones_for_facilities()
    # inherits health's CRS, which is the metric UTM projection set by
    # lga_extractor's clean_layers(), not WGS84. Saving UTM meter
    # coordinates directly into a GeoJSON (a format that's WGS84-only by
    # convention) means downstream web-map consumers (dashboard/app.py's
    # Leaflet overlay) can misread raw UTM values (hundreds of thousands)
    # as if they were tiny lon/lat degrees, causing the map to zoom out
    # to fit an enormous, invalid extent, exactly the "whole map goes
    # blank except a tiny dot" symptom this fixes. grid_access_scored.geojson
    # doesn't need this same fix since `grid` is already in WGS84 by this
    # point in the pipeline, only the isochrones (built from health/roads,
    # which stay in UTM throughout this notebook) were missing it.
    if isochrones.crs is not None and str(isochrones.crs).upper() not in ("EPSG:4326", "OGC:CRS84"):
        isochrones = isochrones.to_crs("EPSG:4326")
    isochrones.to_file(f"{data_dir}/isochrones_health_walk.geojson", driver="GeoJSON")
    print(f"  Done. Saved to {data_dir}/isochrones_health_walk.geojson\n")

### 5.2 Precomputing facility catchments (isochrones) for the dashboard

Beyond the exact travel-time scoring above, it's useful to show a more
intuitive visual on the dashboard: "what area can actually be reached on
foot within 15/30/45 minutes of this specific health facility?" This is
an **isochrone**, a reachable-area polygon, as opposed to the
per-cell travel-time scoring done in Section 5.1.

This is deliberately precomputed here, once, rather than computed live
inside the deployed Streamlit dashboard: building a routable graph and
running isochrone searches takes real time, and a deployed dashboard
should feel instant, not force a visitor to wait through a live OSM
query on every page load. `dashboard/app.py` simply loads this
precomputed file if present and offers it as an optional overlay layer.

**A caveat worth restating from `isochrones.py`'s docstring:** these
polygons are a convex-hull approximation, not the true reachable
street-network footprint, they can overstate actual reachable area,
since real street networks are rarely convex. This is why this overlay
is presented as an illustrative "roughly how far you can walk" layer,
not used anywhere in the project's actual access-deficit *scoring*
(Section 5.1 uses exact network shortest-path routing for that).

### 5.3 Validation and summary statistics

Before moving to the cross-LGA results summary (Notebook 04), it's worth
checking that the scoring behaved sensibly: are underserved percentages in
a plausible range? Does the ranking of modes make sense (walking should
generally show the *highest* underserved percentage, since it's the most
restrictive mode)?

In [ ]:
import pandas as pd

summary_rows = []
for lga_name, grid in scored_results.items():
    settled = grid[grid["building_count"] > 0]
    for mode in MODES:
        col = f"{mode}_access_deficit_score"
        if col not in settled.columns or settled.empty:
            continue
        pct_underserved = 100 * (settled[col] > 0).mean()
        pct_fully_underserved = 100 * (settled[col] == 2).mean()
        summary_rows.append({
            "LGA": lga_name,
            "Mode": mode,
            "Settled cells": len(settled),
            "% underserved (health OR education)": round(pct_underserved, 1),
            "% underserved (health AND education)": round(pct_fully_underserved, 1),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df


If walking doesn't show the highest underserved percentage across both
LGAs, double check the threshold assumptions in Section 3, it likely
means the mode-specific thresholds need adjusting, or (less commonly)
that the road network for that mode resolved unexpectedly (e.g. a sparse
"drive" network in an area with mostly untagged residential roads).

## 6. Visualization

### 6.1 Quick visual check

A fast choropleth-style plot of the walking access-deficit score for one
LGA, as a sanity check that high-deficit cells cluster in plausible
locations (e.g. peri-urban fringes) rather than appearing scattered
randomly across the map, the same "does this look spatially sensible"
check used in Notebook 02, applied here to the routing-based scores
instead of the completeness flags.

As with Notebook 02, this is a throwaway diagnostic plot, not the
polished, publicly shareable visuals built in Notebook 05.

In [ ]:
import matplotlib.pyplot as plt

lga_to_plot = "Akure North"
grid_to_plot = scored_results[lga_to_plot]

fig, ax = plt.subplots(figsize=(8, 8))
settled = grid_to_plot[grid_to_plot["building_count"] > 0]
settled.plot(
    ax=ax, column="walk_access_deficit_score", cmap="RdYlGn_r",
    legend=True, edgecolor="white", linewidth=0.3,
    missing_kwds={"color": "lightgrey"},
)
ax.set_title(f"{lga_to_plot}, Walking Access-Deficit Score")
ax.set_axis_off()
plt.show()


### 6.2 Publication-styled static maps and charts

Beyond the quick visual check above, this produces a full set of
**publication/report-ready static maps and charts**, styled to match a
standard cartographic reference: OSM basemap, lat/long gridlines, north
arrow, scale bar, and legend/colorbar.

For each LGA, this generates:
- An **access-deficit map** per travel mode (walk/okada/drive), using
  the exact same green/amber/red palette as the interactive Streamlit
  map, so a static export and the live dashboard never visually
  disagree.
- A **continuous travel-time map** per service per mode (health and
  education, in minutes), with a colorbar.
- A **data-completeness map** per service, distinguishing "facility
  confirmed nearby" from "possible OSM data gap", the same
  distinction the Findings Summary callout makes in the dashboard.
- A **mode-comparison bar chart**, the static-export equivalent of the
  dashboard's Findings Summary cards.

Every file is saved as a JPEG under `visuals/{lga_name}/`, and the
whole set is zipped for one-click download. These are also read
directly by the Streamlit app's "Static / Publication Maps" section
(`dashboard/app.py`), so regenerating them here keeps the notebook,
the zip download, and the live dashboard all showing the same figures.

**Requires live internet access** for the OSM basemap tiles (works
normally in Colab; if a tile request fails, `add_osm_basemap()`
degrades gracefully to a plain background rather than raising, so the
rest of each figure, data, gridlines, legend, scale bar, still
renders).

In [ ]:
# contextily (OSM basemap tiles for the static maps below) isn't a core
# akure_access dependency, since most of the pipeline doesn't need it,
# it's only required for this publication-map section. Installing it
# explicitly here, same as the reference cartographic notebook's own
# "!pip install -q contextily" pattern, means this section works even
# in a fresh Colab runtime that hasn't run `pip install -e .[static-maps]`.
!pip install -q contextily matplotlib

In [ ]:
import os
import shutil
from akure_access.visualization import generate_all_static_outputs

VISUALS_DIR = "../visuals"
all_static_files = {}  # {lga_name: {"print": [...], "web": [...]}}

for lga_name, grid in scored_results.items():
    lga_dir = f"{VISUALS_DIR}/{lga_name.replace(' ', '_')}"
    print(f"Generating static maps/charts for {lga_name}...")
    # web_dpi=150 produces a second, lighter copy of every figure (from
    # the SAME already-rendered plot, not a re-render) into lga_dir/web/,
    # for fast display inside the Streamlit app; dpi=300 (the default)
    # stays print/download quality for the ZIP in Section 6.3.
    produced = generate_all_static_outputs(lga_name, grid, lga_dir, modes=MODES, dpi=300, web_dpi=150)
    all_static_files[lga_name] = produced
    print(f"  {len(produced['print'])} print-quality + {len(produced['web'])} web-quality files saved to {lga_dir}/\n")

total_print = sum(len(v['print']) for v in all_static_files.values())
total_web = sum(len(v['web']) for v in all_static_files.values())
print(f"Done. {total_print} print-quality + {total_web} web-quality files produced across {len(all_static_files)} LGA(s).")

In [ ]:
# Preview a couple of the generated figures inline (the full set is on
# disk under visuals/ and in the zip below; this is just a sanity check
# that generation worked as expected before moving on). Previewing the
# web-tier copies since they're smaller/faster to render inline here.
from IPython.display import Image, display

preview_lga = list(scored_results.keys())[0]
preview_files = all_static_files[preview_lga]["web"][:2] or all_static_files[preview_lga]["print"][:2]
for f in preview_files:
    print(f)
    display(Image(filename=f))

### 6.3 Download all static maps/charts as one ZIP

Bundles every JPEG produced above (all LGAs, all modes, all layers)
into a single downloadable archive for easy sharing.

In [ ]:
import tempfile

# Build the zip in a TEMP location outside visuals/, then move it in
# afterward, never inside the folder being zipped. Building it directly
# inside VISUALS_DIR (as an earlier version of this cell did) meant
# that on any re-run, the walk could pick up the PREVIOUS run's zip
# file and compress it into the new one too, each re-run getting
# larger and slower, which is almost certainly why this cell ran for
# several minutes rather than the ~10-20 seconds this many JPEGs
# should actually take.
final_zip_path = os.path.join(VISUALS_DIR, "akure_access_static_maps.zip")
if os.path.exists(final_zip_path):
    os.remove(final_zip_path)  # clear out any bloated zip left over from a previous run

with tempfile.TemporaryDirectory() as tmp:
    tmp_zip_base = os.path.join(tmp, "akure_access_static_maps")
    tmp_zip_path = shutil.make_archive(tmp_zip_base, "zip", VISUALS_DIR)
    shutil.move(tmp_zip_path, final_zip_path)

zip_path = final_zip_path
total_files = sum(len(v["print"]) + len(v["web"]) for v in all_static_files.values())
print("Zipped", total_files, "files (print + web tiers) ->", zip_path)
print(f"Archive size: {os.path.getsize(zip_path) / 1e6:.1f} MB")

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except ImportError:
    print("Not running in Colab; find the zip locally at:", zip_path)

## 7. Export

**Outputs produced (per LGA):**

```
data/processed/{lga_name}/grid_access_scored.geojson
data/processed/{lga_name}/isochrones_health_walk.geojson
```

Each grid cell in `grid_access_scored.geojson` carries, per mode
(`walk` / `okada` / `drive`):
- `{service}_time_min_{mode}`, travel time in minutes
- `{service}_distance_km_{mode}`, network distance in km
- `{mode}_access_deficit_score`, composite 0–2 score

This file is what `dashboard/app.py` reads directly, and what Notebook 04
combines across both LGAs for the ranked underserved-settlement list.

`isochrones_health_walk.geojson` carries one row per (health facility,
trip time) combination, with columns `facility_name`, `osmid`,
`trip_time_min` (15/30/45), and a convex-hull reachable-area polygon.
`dashboard/app.py` loads this as an optional overlay, see its own
docstring for how it handles this file being absent (e.g. for an LGA
where this notebook hasn't been re-run since this feature was added).

## 8. Summary

**What this notebook accomplished:** built routable road networks for
three transport modes, computed nearest-facility travel time to health
and education services for every settled grid cell in both LGAs, and
derived a composite access-deficit score per mode, using a multi-source
Dijkstra approach for tractable runtime at this project's scale.

**Outputs produced:** see Section 7 above.

**Next notebook:** `04_results_summary.ipynb` combines both LGAs' scored
grids, produces the ranked underserved-settlement list, and exports
summary tables for the project report and StoryMap.